# Compression / codec impact

Fixed `N = 1_000_000` point cloud.  Sweep the `compressor=` kwarg
(added in the production refactor) across five values and measure
write time, read time, and disk MB.

| Label | `compressor=` value |
|-------|--------------------|
| `none` | `'none'` — uncompressed chunks |
| `zstd` (default) | `None` — zarr v3's default |
| `blosc_l1` | `[{...}]` Blosc + Zstd level 1, no shuffle |
| `blosc_l5` | `[{...}]` Blosc + Zstd level 5, no shuffle |
| `blosc_l5_bitshuffle` | `'blosc'` — Blosc + Zstd level 5 + BitShuffle |

Compression CPU usually pays off on disk; bitshuffle is expected to
beat plain Zstd on float32 positions.  See
[`docs/spec/foundations/codec_pipeline.md`](../docs/spec/foundations/codec_pipeline.md)
for the kwarg semantics.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points

N      = 1_000_000
CHUNK  = (200.0, 200.0, 200.0)
BIN    = (50.0,  50.0,  50.0)
SEED   = 0

CONFIGS = [
    ('none',                'none'),
    ('zstd',                None),
    ('blosc_l1',            [{'name': 'blosc', 'configuration': {
                                'cname': 'zstd', 'clevel': 1,
                                'shuffle': 'noshuffle',
                                'typesize': 4, 'blocksize': 0}}]),
    ('blosc_l5',            [{'name': 'blosc', 'configuration': {
                                'cname': 'zstd', 'clevel': 5,
                                'shuffle': 'noshuffle',
                                'typesize': 4, 'blocksize': 0}}]),
    ('blosc_l5_bitshuffle', 'blosc'),
]

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)
positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)

metrics = ['write_s', 'read_all_s']
raw = {m: np.zeros((len(CONFIGS), N_RUNS)) for m in metrics}
disk_MB = np.zeros(len(CONFIGS))

for i, (label, compressor) in enumerate(CONFIGS):
    for run in range(N_RUNS):
        store = _new_store(f'codec_{label}_run{run}')
        t_w, _ = _time(
            write_points, store, positions,
            chunk_shape=CHUNK, bin_shape=BIN, compressor=compressor,
        )
        t_r, _ = _time(read_points, store)
        raw['write_s'][i, run]    = t_w
        raw['read_all_s'][i, run] = t_r
        if run == 0:
            disk_MB[i] = _store_bytes(store) / 1e6
        shutil.rmtree(Path(store).parent, ignore_errors=True)

rows = []
for i, (label, _) in enumerate(CONFIGS):
    row = {'config': label, 'disk_MB': round(disk_MB[i], 2)}
    for m in metrics:
        mean, hw = _mean_ci95(raw[m][i])
        row[f'{m}_mean'] = round(mean, 4)
        row[f'{m}_hw']   = round(hw,   4)
    rows.append(row)
df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
x = np.arange(len(df))
ROT = 20

# Panel 1: write time.
ax = axes[0]
ax.bar(x, df['write_s_mean'], yerr=df['write_s_hw'],
       capsize=3, color='C0')
ax.set_xticks(x); ax.set_xticklabels(df['config'], rotation=ROT, ha='right')
ax.set_ylabel('s')
ax.set_title('Write time')
ax.grid(True, axis='y', alpha=0.3)

# Panel 2: read all time.
ax = axes[1]
ax.bar(x, df['read_all_s_mean'], yerr=df['read_all_s_hw'],
       capsize=3, color='C1')
ax.set_xticks(x); ax.set_xticklabels(df['config'], rotation=ROT, ha='right')
ax.set_ylabel('s')
ax.set_title('Read all')
ax.grid(True, axis='y', alpha=0.3)

# Panel 3: disk MB.
ax = axes[2]
ax.bar(x, df['disk_MB'], color='C3')
ax.set_xticks(x); ax.set_xticklabels(df['config'], rotation=ROT, ha='right')
ax.set_ylabel('MB')
ax.set_title('Disk size')
ax.grid(True, axis='y', alpha=0.3)

ratio = df.loc[df['config'] == 'none', 'disk_MB'].iloc[0] / \
        df.loc[df['config'] == 'blosc_l5_bitshuffle', 'disk_MB'].iloc[0]
fig.suptitle(
    f'Compression impact — point cloud (N = {N:,}, '
    f'{N_RUNS} runs, 95% CI) — blosc bitshuffle gives {ratio:.1f}× smaller disk',
)
plt.tight_layout()